# Fine-tune v2 — fixes applied

**Run this on Google Colab (GPU runtime, e.g. Tesla T4) — it will NOT run on a machine without an NVIDIA GPU, because bitsandbytes 4-bit quantization requires CUDA.**

Changes vs the original notebook:
1. `TRAIN_FILE` now points to the merged `train.jsonl` (247 examples = original 47 + 200 new ones) — upload this file to the Colab session.
2. LoRA `target_modules` now includes MLP layers (`gate_proj`, `up_proj`, `down_proj`) in addition to attention layers — factual knowledge lives mostly in MLP layers, so training only attention layers capped accuracy at ~63% no matter how long training ran.
3. `r` increased 16 -> 32 (more layers are being trained now, so a bit more capacity is used).
4. Eval set is now a proper 10% held-out random split of the full 247-example dataset (`train_test_split`), instead of the old static 5-example `eval.jsonl`.
5. `load_best_model_at_end=True` + `EarlyStoppingCallback` — last time, validation loss was best at epoch 3 (1.74) then got worse every epoch up to epoch 9 (2.13) while training loss kept dropping — that's overfitting. This time the trainer keeps the best checkpoint automatically instead of the last (most overfit) one.
6. The before/after comparison now loads a **fresh, separate base model instance** for the "before" output, instead of reusing the already-fine-tuned `model` object (which was the bug in the original notebook — it printed the same fine-tuned output for both "before" and "after").

In [ ]:
!pip install -U -q bitsandbytes accelerate transformers peft trl datasets

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, EarlyStoppingCallback
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
TRAIN_FILE = "train.jsonl"  # upload this to the Colab session first
OUTPUT_DIR = "qwen2.5-1.5b-trst01-compliance-v2"

In [ ]:
# carve out a proper held-out eval split instead of just eyeballing training loss
full_dataset = load_dataset("json", data_files={"train": TRAIN_FILE})["train"]
dataset = full_dataset.train_test_split(test_size=0.1, seed=42)
print(dataset)

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected - stop here, this notebook needs a GPU runtime")

In [ ]:
# --- Load base model in 4-bit (QLoRA) ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False

In [ ]:
# LoRA config: now includes MLP layers, not just attention
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
)

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=15,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    bf16=True,
    max_length=1024,
    packing=False,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=3,
)

In [ ]:
# --- Train ---
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

In [ ]:
# --- Save adapter locally (this is the BEST checkpoint, thanks to load_best_model_at_end) ---
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved LoRA adapter to {OUTPUT_DIR}")

In [ ]:
# --- Real before/after comparison: base model loaded FRESH (separate instance), not reusing the trained `model` object ---
test_questions = [
    "What is EUDR and which companies does it apply to?",
    "What is the difference between mass balance and identity preserved chain of custody?",
]

def generate(m, tok, prompt):
    messages = [{"role": "user", "content": prompt}]
    inputs = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(m.device)
    out = m.generate(**inputs, max_new_tokens=200, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("=== BASE MODEL (fresh, untrained) OUTPUT ===")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
for q in test_questions:
    print(f"\nQ: {q}\nA: {generate(base_model, tokenizer, q)}")

del base_model
torch.cuda.empty_cache()

print("\n=== FINE-TUNED MODEL OUTPUT ===")
for q in test_questions:
    print(f"\nQ: {q}\nA: {generate(model, tokenizer, q)}")

# Push the trained adapter to Hugging Face

This was missing before — the adapter was only being saved locally to `OUTPUT_DIR`, never actually pushed to the Hub. That's why `sajjan0001/qwen2.5-1.5b-trst01-compliance` only has the tokenizer files and no `adapter_config.json` / `adapter_model.safetensors`.

Run the login cell once (it'll ask for a token with **write** access — get one from https://huggingface.co/settings/tokens), then push.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()  # paste a token with WRITE access

In [ ]:
REPO_ID = "sajjan0001/qwen2.5-1.5b-trst01-compliance"

# use model.push_to_hub(), not trainer.push_to_hub() - the latter's first arg is a
# commit message, not a repo id, which is what sent the adapter to the wrong repo last time
model.push_to_hub(REPO_ID)
tokenizer.push_to_hub(REPO_ID)
print(f"Pushed adapter to https://huggingface.co/{REPO_ID}")